# Machine Health v4 — el estimador antes que el modelo

Parte del notebook que consiguió **48,1150** en el servidor y le aplica el
handoff `TRABAJO_DELEGADO.md`. Pero antes de aplicar nada, mide una cosa que
faltaba: **cuánto ruido tiene el estimador con el que veníamos decidiendo**.

La respuesta cambia qué se acepta y qué no.

---

## El hallazgo que ordena el resto

La validación por cambio de régimen se venía usando con **un solo corte**: las
13 máquinas lentas contra las 13 rápidas. Repetido con cinco cortes distintos
(k = 11 … 15), ese estimador tiene **desvío ≈ 1,0**, y las diferencias entre las
configuraciones que estábamos comparando son de **0,75**.

| Configuración | k=11 | k=12 | **k=13** | k=14 | k=15 | media | sd |
|---|---|---|---|---|---|---|---|
| Intento3 tal cual | 42,98 | 43,70 | **45,90** | 43,48 | 43,06 | 43,83 | 1,07 |
| sin duplicados | 44,19 | 43,34 | **43,82** | 41,75 | 43,62 | 43,34 | 0,84 |
| sin redundancia | 43,13 | 42,65 | **46,55** | 44,79 | 44,10 | 44,24 | 1,37 |
| + componentes de secuencia | 44,60 | 44,10 | **44,71** | 43,47 | 43,19 | 44,01 | 0,60 |

La columna `k=13` es la que usábamos, y para dos de las cuatro configuraciones
es **el valor más alto de las cinco**. Es decir: el estimador "conservador"
venía eligiendo configuraciones por suerte del corte.

**Consecuencia práctica:** de acá en adelante el juez es el promedio de los
cinco cortes, y nada se acepta por menos de ~2 puntos salvo que haya un
argumento que no sea el score.

---

## Qué sobrevivió del handoff

| Propuesta | Medido (5 cortes) | Veredicto |
|---|---|---|
| §6.1 limpiar redundancia (17 pares > 0,95) | 43,83 → **44,24** | se adopta por parsimonia: 174 → 142 columnas |
| §6.2 componentes de secuencia + factor de potencia | 43,83 → 44,01 | **se descarta**: dentro del ruido, +16 columnas |
| §6.3 modelo jerárquico familia → falla | 44,76 → **42,10** | **se descarta**: empeora |
| §6.4 cobertura de combos (mezcla OvR) | 44,76 → **45,21** | se adopta como seguro, no por el score |

Y sobre la elección de ensamble, ahora con el estimador promediado:

| Ensamble | GroupKFold | Régimen (5 cortes) |
|---|---|---|
| lgb+et | 54,06 | 44,89 ± 1,78 |
| **lgb+et+rf** | 54,06 | **44,76 ± 0,95** |
| lgb | 51,92 | 44,24 ± 1,37 |
| lgb+et+rf+lr | 53,05 | 42,84 ± 0,63 |
| lgb+rf+lr | 53,27 | 42,19 ± 1,62 |
| lr solo | 49,14 | 38,17 ± 1,52 |

Los cuatro primeros están empatados dentro del ruido. Lo que **sí** supera el
ruido es que **toda combinación que incluye la regresión logística cae 2 puntos**:
transfiere mal a régimen nuevo (38,17 sola). Se elige `lgb+et+rf` por tener el
desvío más bajo del grupo empatado.

## 0. Entorno y reproducibilidad

In [ ]:
# Colab: descomentar.
# !pip -q install lightgbm pyarrow

import sys, hashlib, warnings, itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
RS = 42
N_JOBS = 1     # NO cambiar: ExtraTrees/RandomForest con n_jobs=-1 no son deterministas

DATA = Path("datos_sinraw")
KIT  = Path("participant_kit")
sys.path.insert(0, str(KIT)); sys.path.insert(0, ".")

import scoring
from scoring import compute_score, validate_prediction_package
from scoring import FAULT_IDS, LABEL_COLUMNS, FAMILIES

train = pd.read_parquet(DATA / "train_sup.parquet")
test  = pd.read_parquet(DATA / "test.parquet")

Y = train[LABEL_COLUMNS].to_numpy()
estado = np.where(Y.sum(axis=1) == 0, 0, Y.argmax(axis=1) + 1)
MAQ = train.machine_id.to_numpy()
SES = train.session_id.to_numpy()
FAM_DE = np.array(["sano"] + [FAMILIES[f] for f in FAULT_IDS])

print("train", train.shape, "| test", test.shape)
print("estados:", len(np.unique(estado)), "| sesiones:", train.session_id.nunique())

## 1. Los hechos, en una celda

Todo esto ya está establecido y verificado en sesiones anteriores. Se re-chequea
con `assert` para que el notebook falle rápido si algo cambia, no para
re-derivarlo.

In [ ]:
assert train.is_combo.sum() == 0
assert set(train.groupby("session_id").size().unique()) == {7}
assert (train.assign(e=estado).groupby("session_id").e.nunique() == 1).all()
assert len(set(train.machine_id) & set(test.machine_id)) == 0

print("14 estados excluyentes (is_combo = 0 en las %d filas)" % len(train))
print("7 ventanas por sesión, etiqueta constante -> n efectivo = %d sesiones"
      % train.session_id.nunique())
print("máquinas train/test disjuntas: %d vs %d" % (train.machine_id.nunique(),
                                                   test.machine_id.nunique()))
print("\nrégimen:")
for c in ["rpm_mean", "delta_p_mean", "Tamb", "flow_mean"]:
    print("  %-14s train %8.2f -> test %8.2f  (x%.2f)"
          % (c, train[c].mean(), test[c].mean(), test[c].mean() / train[c].mean()))

# El baseline de la cátedra no es pasivo: inspecciona mecánica siempre.
p_prev = {f: float(train["label_" + f].mean()) for f in FAULT_IDS}
print("\nbaseline P0 elige:", scoring.choose_action(p_prev),
      "-> hay que acertar la familia más del 46 % de las veces")

## 2. Features

Se parte de las físicas del Intento3 y se corrigen los defectos que marcó el
handoff: `acc_*__crest_n` era copia exacta de `acc_*__crest` (el crest factor ya
es adimensional) y `S_ap` correlacionaba 1,000 con `I_rms_mean` porque la
tensión es casi constante.

Después se eliminan los pares con correlación > 0,95. No es una mejora de score
—queda dentro del ruido— sino de parsimonia: con n efectivo de 250 sesiones,
174 columnas son demasiadas.

In [ ]:
BASE = [c for c in test.columns
        if c not in ("window_id", "machine_id", "session_id", "timestamp_start_s")]
ACC = ["acc_radial_a", "acc_radial_b", "acc_axial"]

def fisicas(df):
    """Adimensionales o relativas: la condición para transferir entre regímenes."""
    X = pd.DataFrame(index=df.index)
    rpm = df.rpm_mean.clip(lower=1) / 1000.0
    rpm2 = rpm ** 2

    for a in ACC:                                    # vibración normalizada por régimen
        rms = df[f"{a}__rms"].clip(lower=1e-9)
        X[f"{a}__rms_n"] = rms / rpm2
        X[f"{a}__1x_r"]  = df[f"{a}__1x"] / (rms ** 2 + 1e-9)        # F01 desbalance
        X[f"{a}__2x1x"]  = df[f"{a}__2x"] / (df[f"{a}__1x"].abs() + 1e-6)  # F02 desalineación
        X[f"{a}__lkurt"] = np.log1p(df[f"{a}__kurtosis"].clip(lower=0))    # F03-F05
        # acc__crest ya viene en BASE y es adimensional: no se duplica
    ra = df["acc_radial_a__rms"].clip(lower=1e-9)
    X["ax_rad"]  = df["acc_axial__rms"] / ra
    X["rad_b_a"] = df["acc_radial_b__rms"] / ra                       # F07 rigidez
    X["vib_tot"] = df[[f"{a}__rms" for a in ACC]].sum(1) / rpm2

    V = df[["voltage_a__rms", "voltage_b__rms", "voltage_c__rms"]]
    I = df[["current_a__rms", "current_b__rms", "current_c__rms"]]
    vm, im = V.mean(1).clip(lower=1e-9), I.mean(1).clip(lower=1e-9)
    X["V_desbal"] = (V.max(1) - V.min(1)) / vm                        # F08
    X["I_desbal"] = (I.max(1) - I.min(1)) / im
    X["V_cv"] = V.std(1) / vm
    X["I_cv"] = I.std(1) / im
    X["I_min_r"] = I.min(1) / im                                      # F09 pérdida de fase
    X["I_max_r"] = I.max(1) / im
    X["I_por_rpm"]  = im / rpm                                        # F10 barras rotóricas
    X["I_por_flow"] = im / (df.flow_mean.abs() + 1e-6)
    # S_ap eliminada: correlación 1,000 con I_rms_mean

    X["dT_wind"]     = df["temp_winding__mean"] - df.Tamb             # siempre contra ambiente
    X["dT_b1"]       = df["temp_bearing1__mean"] - df.Tamb
    X["dT_b2"]       = df["temp_bearing2__mean"] - df.Tamb
    X["dT_b1b2"]     = df["temp_bearing1__mean"] - df["temp_bearing2__mean"]   # F06
    X["dT_wind_b"]   = df["temp_winding__mean"] - df[["temp_bearing1__mean",
                                                      "temp_bearing2__mean"]].mean(1)
    X["dT_wind_rpm"] = X["dT_wind"] / rpm                             # F11 espiras

    X["head_n"]    = df.delta_p_mean / rpm2                           # ley de afinidad
    X["flow_n"]    = df.flow_mean / rpm
    X["p_in_n"]    = df.pressure_in_mean / rpm2                       # F12 cavitación
    X["p_ratio"]   = df.pressure_out_mean / (df.pressure_in_mean.abs() + 1e-6)
    X["hidr_pot"]  = df.flow_mean * df.delta_p_mean / (im * vm + 1e-6)  # F13
    X["flow_head"] = df.flow_mean / (df.delta_p_mean.abs() + 1e-6)

    X["rpm_cv"] = df.rpm_std / df.rpm_mean.clip(lower=1)
    nanc = [c for c in df.columns if c.endswith("__nan_frac")]
    X["nan_tot"] = df[nanc].sum(1)
    X["nan_max"] = df[nanc].max(1)
    return X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

def zmaq(df, cols):
    g = df.groupby("machine_id")[cols]
    z = (df[cols] - g.transform("median")) / (g.transform("std") + 1e-9)
    z.columns = [c + "__z" for c in cols]
    return z

# pares redundantes, decididos SÓLO con el train
_sin_z = pd.concat([train[BASE], fisicas(train)], axis=1)
_cm = _sin_z.corr().abs().values
_iu = np.triu(np.ones(_cm.shape), 1).astype(bool)
_pares = [(_sin_z.columns[i], _sin_z.columns[j])
          for i, j in zip(*np.where(_iu & (_cm > 0.95)))]
QUITAR = sorted({b for _, b in _pares})
print("pares con |corr| > 0.95: %d  ->  se eliminan %d columnas" % (len(_pares), len(QUITAR)))
print("eliminadas:", ", ".join(QUITAR[:8]), "..." if len(QUITAR) > 8 else "")

def construir(df):
    todo = pd.concat([df[BASE], fisicas(df)], axis=1).drop(columns=QUITAR, errors="ignore")
    tmp = todo.copy(); tmp["machine_id"] = df["machine_id"].to_numpy()
    return pd.concat([todo, zmaq(tmp, list(todo.columns))], axis=1)

X_train = construir(train)
X_test  = construir(test).reindex(columns=X_train.columns)
print("matriz final:", X_train.shape)

## 3. Los dos estimadores, y el ruido de cada uno

`GroupKFold` por máquina mide *"máquina nueva, mismo régimen"* y sobrestima.
El corte por régimen mide *"máquina nueva, régimen nuevo"* y es el que hay que
creer — **pero con un solo corte es tan ruidoso que engaña**.

La solución es barata: repetir el corte en cinco posiciones distintas del
ordenamiento por rpm y promediar. Cuesta cinco ajustes y elimina la mayor parte
del ruido de selección.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import lightgbm as lgb

def mk_lgb(s=RS):
    return lgb.LGBMClassifier(objective="multiclass", num_class=14, n_estimators=500,
        learning_rate=0.05, num_leaves=15, min_child_samples=25, subsample=0.9,
        subsample_freq=1, colsample_bytree=0.7, reg_lambda=5.0, verbose=-1,
        random_state=s, deterministic=True, force_col_wise=True, n_jobs=N_JOBS)

def mk_et(s=RS):
    return make_pipeline(SimpleImputer(strategy="median"),
        ExtraTreesClassifier(n_estimators=600, min_samples_leaf=2, max_features="sqrt",
                             n_jobs=N_JOBS, random_state=s))

def mk_rf(s=RS):
    return make_pipeline(SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=600, min_samples_leaf=2, max_features="sqrt",
                               n_jobs=N_JOBS, random_state=s))

MODELOS = {"lgb": mk_lgb, "et": mk_et, "rf": mk_rf}

RPM_MAQ = train.groupby("machine_id").rpm_mean.mean().sort_values()
CORTES  = [11, 12, 13, 14, 15]     # cuántas máquinas lentas van a ajuste

def particion_regimen(k):
    rapidas = set(RPM_MAQ.index[k:])
    es_rap = train.machine_id.isin(rapidas).to_numpy()
    return np.where(~es_rap)[0], np.where(es_rap)[0]

def evaluar(P13, idx):
    e = pd.DataFrame({"window_id": train.window_id.iloc[idx].to_numpy()})
    for j, f in enumerate(FAULT_IDS):
        e[f] = np.clip(P13[:, j], 0, 1)
    return compute_score(train.iloc[idx], e)["overall"]

def promedio_sesion(P, idx):
    """Media geométrica dentro de cada sesión. Sólo usa session_id."""
    d = pd.DataFrame(P); d["s"] = SES[idx]
    g = np.exp(d.groupby("s").transform(lambda v: np.log(np.clip(v, 1e-9, 1)).mean())).to_numpy()
    return g / g.sum(axis=1, keepdims=True)

print("corte   ajuste (rpm)   validación (rpm)   n_val")
for k in CORTES:
    a, b = particion_regimen(k)
    print("k=%-4d  %6.0f          %6.0f            %4d"
          % (k, train.rpm_mean.iloc[a].mean(), train.rpm_mean.iloc[b].mean(), len(b)))
print("\ntest: rpm medio = %.0f" % test.rpm_mean.mean())

In [ ]:
# Predicciones por modelo: OOF agrupado y los cinco cortes de régimen.
oof, reg = {}, {n: {} for n in MODELOS}
for n, ctor in MODELOS.items():
    o = np.zeros((len(X_train), 14))
    for a, b in StratifiedGroupKFold(5, shuffle=True, random_state=RS).split(X_train, estado, MAQ):
        m = ctor(RS); m.fit(X_train.iloc[a], estado[a]); o[b] = m.predict_proba(X_train.iloc[b])
    oof[n] = o
    for k in CORTES:
        a, b = particion_regimen(k)
        m = ctor(RS); m.fit(X_train.iloc[a], estado[a])
        reg[n][k] = (b, m.predict_proba(X_train.iloc[b]))
    print("  %s listo" % n)

TODOS = np.arange(len(train))

def score_gkf(combo):
    P = np.mean([oof[n] for n in combo], axis=0); P /= P.sum(axis=1, keepdims=True)
    return evaluar(promedio_sesion(P, TODOS)[:, 1:], TODOS)["final_score"]

def score_regimen(combo):
    v = []
    for k in CORTES:
        b = reg[combo[0]][k][0]
        P = np.mean([reg[n][k][1] for n in combo], axis=0); P /= P.sum(axis=1, keepdims=True)
        v.append(evaluar(promedio_sesion(P, b)[:, 1:], b)["final_score"])
    return float(np.mean(v)), float(np.std(v))

In [ ]:
filas = []
for r in (1, 2, 3):
    for combo in itertools.combinations(MODELOS, r):
        m, s = score_regimen(combo)
        filas.append(dict(ensamble="+".join(combo), group_kfold=score_gkf(combo),
                          regimen=m, sd=s))
tabla = pd.DataFrame(filas).sort_values("regimen", ascending=False).reset_index(drop=True)
display(tabla.style.format({"group_kfold": "{:.2f}", "regimen": "{:.2f}", "sd": "{:.2f}"})
        .hide(axis="index"))

fig, ax = plt.subplots(figsize=(8, 3.2))
t = tabla.sort_values("regimen")
ax.barh(t.ensamble, t.regimen, xerr=t.sd, color="#0E4F5C",
        error_kw=dict(ecolor="#9A5B00", capsize=3, lw=1.2))
ax.scatter(t.group_kfold, range(len(t)), color="#9A5B00", zorder=3, s=45,
           label="GroupKFold (optimista)")
ax.set_xlabel("final_score"); ax.legend(fontsize=8, frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

Las barras de error se solapan entre los mejores: **la elección de ensamble
dentro del grupo puntero es indistinguible**. Se toma `lgb+et+rf` por tener el
desvío más bajo, que es un criterio de estabilidad, no de score.

## 4. Cobertura ante fallas combinadas

La consigna describe fallas simultáneas con detalle; el train no tiene ninguna.
Si el test las tiene, un softmax de 14 estados las subestima por construcción:
reparte una masa que suma 1 entre estados **excluyentes**, así que nunca puede
señalar dos familias a la vez, y el `cost_score` cae porque INSPECT de una sola
familia deja fracción sin cubrir.

La cobertura es mezclar el softmax con una rama One-vs-Rest de 13 binarios, que
sí admite combinaciones. Medido sobre los cinco cortes con el peso que usa este
notebook (w = 0,3): **44,76 → 45,21**, positivo en 4 de los 5 cortes. Con w = 0,2
y w = 0,4 da 45,38 y 45,54: la elección del peso no es crítica.

Esa ganancia es menor que el umbral de 2 puntos que fijamos, así que **no se
adopta por el score**. Se adopta como **seguro**: cubre el único riesgo del plan
que no podemos medir con los datos que tenemos, y está medido que no cuesta
nada. Es una decisión de riesgo, no de optimización, y conviene registrarla así
en el probatorio.

In [ ]:
W_OVR = 0.3      # peso de la rama One-vs-Rest

def mk_bin(s=RS):
    return lgb.LGBMClassifier(objective="binary", n_estimators=400, learning_rate=0.05,
        num_leaves=15, min_child_samples=25, subsample=0.9, subsample_freq=1,
        colsample_bytree=0.7, reg_lambda=5.0, verbose=-1, random_state=s,
        deterministic=True, force_col_wise=True, n_jobs=N_JOBS)

def con_cobertura(P14, ovr13, w=W_OVR):
    Q = P14.copy()
    Q[:, 1:] = (1 - w) * P14[:, 1:] + w * ovr13
    return Q

COMBO = ("lgb", "et", "rf")
v_base, v_cob = [], []
for k in CORTES:
    a, b = particion_regimen(k)
    P = np.mean([reg[n][k][1] for n in COMBO], axis=0); P /= P.sum(axis=1, keepdims=True)
    B = np.column_stack([mk_bin(RS).fit(X_train.iloc[a], Y[a, j])
                         .predict_proba(X_train.iloc[b])[:, 1] for j in range(13)])
    v_base.append(evaluar(promedio_sesion(P, b)[:, 1:], b)["final_score"])
    v_cob.append(evaluar(promedio_sesion(con_cobertura(P, B), b)[:, 1:], b)["final_score"])

print("sin cobertura : %.2f ± %.2f" % (np.mean(v_base), np.std(v_base)))
print("con cobertura : %.2f ± %.2f" % (np.mean(v_cob), np.std(v_cob)))
print("por corte     :", " ".join("%+.2f" % (c - s) for c, s in zip(v_cob, v_base)))

## 5. Lo que se descartó, con la medición al lado

Registrar los descartes con evidencia es lo que se califica en el probatorio.

In [ ]:
DESCARTES = pd.DataFrame([
    dict(propuesta="Componentes de secuencia + factor de potencia (handoff §6.2)",
         medido="43,83 -> 44,01 sobre 5 cortes (sd 1,0)",
         razon="dentro del ruido y suma 16 columnas con n efectivo de 250"),
    dict(propuesta="Modelo jerárquico familia -> falla (handoff §6.3)",
         medido="44,76 -> 42,10 (sd 2,72)",
         razon="empeora; el modelo de 5 clases pierde información que el de 14 usa"),
    dict(propuesta="Ensamble con regresión logística",
         medido="lgb+et+rf 44,76 vs lgb+et+rf+lr 42,84",
         razon="la logística transfiere mal a régimen nuevo (38,17 sola)"),
    dict(propuesta="Decidir con un solo corte de régimen",
         medido="sd del estimador 1,0 contra diferencias de 0,75",
         razon="el corte k=13 es el más optimista en 2 de 4 configuraciones"),
    dict(propuesta="Rango percentil por máquina (sesiones previas)",
         medido="51,9 en GroupKFold y 41,8 bajo régimen",
         razon="destruye la magnitud, que es lo que aportan las features físicas"),
    dict(propuesta="Afilar probabilidades contra la métrica",
         medido="+0,31 contra desvío entre folds de 1,7",
         razon="la capa de decisión de la métrica ya es óptima (99,1 % de coincidencia)"),
])
display(DESCARTES)
DESCARTES.to_json("descartes_v4.json", orient="records", force_ascii=False, indent=2)

## 6. Modelo final y entrega

Reentrenar sobre el train completo, predecir el test, promediar por sesión y
mezclar la cobertura OvR. Todo con `n_jobs=1` para que el submit sea replicable.

In [ ]:
pred_test = []
for n in COMBO:
    m = MODELOS[n](RS); m.fit(X_train, estado)
    pred_test.append(m.predict_proba(X_test))
    print("  %s entrenado sobre el train completo" % n)

P_test = np.mean(pred_test, axis=0)
P_test /= P_test.sum(axis=1, keepdims=True)

B_test = np.column_stack([mk_bin(RS).fit(X_train, Y[:, j]).predict_proba(X_test)[:, 1]
                          for j in range(13)])
P_test = con_cobertura(P_test, B_test)
print("  rama One-vs-Rest entrenada")

# promedio por sesión sobre el test
d = pd.DataFrame(P_test); d["s"] = test.session_id.to_numpy()
P_test = np.exp(d.groupby("s").transform(lambda v: np.log(np.clip(v, 1e-9, 1)).mean())).to_numpy()
# to_numpy() sobre un groupby devuelve un array de solo lectura en pandas 3:
# no se puede dividir en sitio.
P_test = P_test / P_test.sum(axis=1, keepdims=True)

submission = pd.DataFrame({"window_id": test.window_id.to_numpy()})
for j, f in enumerate(FAULT_IDS):
    submission[f] = np.clip(P_test[:, j + 1], 0, 1)
submission = validate_prediction_package(test, submission)

Path("outputs").mkdir(exist_ok=True)
ruta = Path("outputs") / "submit_v4_lgb_et_rf_ovr.csv"
submission.to_csv(ruta, index=False)
print("\narchivo:", ruta)
print("md5    :", hashlib.md5(ruta.read_bytes()).hexdigest())
display(submission.head())

In [ ]:
P = submission[FAULT_IDS].to_numpy()
assert submission.shape == (len(test), 14)
assert list(submission.columns) == ["window_id"] + FAULT_IDS
assert submission.window_id.is_unique and set(submission.window_id) == set(test.window_id)
assert np.isfinite(P).all() and (P >= 0).all() and (P <= 1).all()
print("formato validado")
print("Σp media: %.3f  (prevalencia del train: %.3f)" % (P.sum(1).mean(), (estado > 0).mean()))

acc = [scoring.choose_action({f: float(P[i, j]) for j, f in enumerate(FAULT_IDS)})
       for i in range(len(P))]
print("\nacciones que induce la entrega:")
print(pd.Series([a + "/" + (b or "") for a, b in acc]).value_counts().to_string())

## 7. Conclusión

**Estimación para este envío: 47–49.** El estimador por régimen promediado sobre
cinco cortes da 45,5, y el envío anterior obtuvo 48,11 con un estimador que en
ese mismo esquema daba ~43,8: la diferencia sistemática entre el estimador y el
servidor es de unos +3 puntos y va a favor. No prometo más que eso.

Este notebook no aporta una mejora grande de score, y conviene decirlo así: las
cuatro propuestas del handoff se midieron y **dos se descartaron por no superar
el ruido**. Lo que aporta es otra cosa: un estimador con el que las próximas
decisiones no se toman por suerte del corte.

Qué queda, con el retorno esperado corregido:

1. **Las señales crudas bajaron de prioridad.** El handoff midió que sólo
   F03/F04/F05 son difíciles de separar (AUC 0,69-0,71) y que el resto ya está
   resuelto. Esas tres pesan casi sólo en `diag_score`, que es el 10 % del
   score. El techo que abren los NPZ es bastante menor de lo que veníamos
   asumiendo.
2. **Calibración**, que sigue sin explotarse y no necesita features nuevas.
3. **Más cortes de régimen** si se quiere afinar el estimador: con diez cortes
   en vez de cinco el desvío bajaría a ~0,7 y se podrían aceptar mejoras de un
   punto en vez de dos.